# 3. Ask, run, inspect and repair

**Open:** the project, restart R, and Save As `scripts/my-03-working-with-ai.R` from the supplied script.

**Task:** repair code that stops with an error, then check code that runs but changes the requested analysis. Report the mean of the **recorded** counts without filling the missing result. Save the evidence and a short AI-use log.

The AI turns below are illustrative, not a real chat transcript. The R error and numerical outputs are generated by the supplied code. Choose a browser chat from [AI access](01-first-run.ipynb#id-1-5-ai-access) for a live attempt, or complete the supplied dialogue as a code-inspection route. Both routes include executable R checks; record the route you actually use.

**Recommended order:** first complete Sections 3.1–3.6 using the supplied exercises. Then try the live-chat activity in Section 3.7. In your my- script, keep the supplied error examples for comparison and add your own repairs in the separate **live-workspace** block at the end.

## 3.1 Give the assistant something it can actually use

Run `context`:

In [1]:
practice <- read.csv("data/practice_counts.csv", stringsAsFactors = FALSE)
names(practice)
practice

[1] "sample_id"     "count_cfu_ml"  "storage_hours"

sample_id,count_cfu_ml,storage_hours
<chr>,<int>,<int>
P01,10,0
P02,20,2
P03,30,4
P04,NA,6


A chat assistant does not automatically receive your RStudio object. For the later live activity, copy the column names and four-row output from the Console into chat. These rows are invented teaching data; share only approved information in a real project.

````{admonition} You → AI: the request
:class: student-turn
```text
I am learning base R in RStudio. My object practice contains one row per
invented milk sample, with columns sample_id, count_cfu_ml and storage_hours.
I will paste the column names and table below.

Report the missing count, recorded count and mean of the recorded counts.
Leave missing values unchanged. Give one small R block, explain its output,
and suggest a hand check. Do not install packages or change files.
```
````

Read this as a model request while completing the reference exercise. Section 3.7 tells you exactly where to put an actual reply in your my- script.

## 3.2 Repair code that stops

An assistant in a **new chat** has not seen your object or table. This incomplete request illustrates that problem; you may try it during Section 3.7:

````{admonition} You → AI: a request missing context
:class: student-turn
```text
I have already loaded my milk-count data into R.
Write code to calculate the mean concentration, ignoring missing values.
```
````

An assistant might guess an object called `milk_data`. It might also ask for your actual names, which is a useful response. A vague request does not guarantee an error; the following constructed case gives everyone the same failure to inspect.

**Make R show the error:** restart R and rerun `context` first, so objects created by a live attempt cannot change this exercise. In your my- script's `runtime-error` block, select **only** this line and Run:

```r
mean(milk_data$count_cfu_ml, na.rm = TRUE)
```

R reports `object 'milk_data' not found` and stops that calculation. Your session still contains `practice`; the error does not erase it. The code asks for an object with the wrong name, so R never reaches the mean calculation.

Now run the **whole supplied `runtime-error` block** to capture that actual error for your saved record. Its supplied recording code lets the script continue after this expected failure; you do not need to write it yourself.

**Keep this case as the error example.** Put the corrected calculation in `runtime-repair` or in your live-work area, rather than replacing the failing line inside `runtime-error`. Preserve its original error record for comparison with the repair. If you already changed the example, its recording block now explains that no error was observed; it does not invent an error or raise a second assertion. You can restore just this block from the downloaded reference script.

In [2]:
# To see R stop in the Console, select ONLY the mean(...) line and Run.
# Then run this whole supplied block to capture the error for the lesson record.
# The guard also lets the complete script run through this expected failure.
runtime_failure <- tryCatch({
  mean(milk_data$count_cfu_ml, na.rm = TRUE)
}, error = function(e) e)
runtime_error_observed <- inherits(runtime_failure, "error")
if (runtime_error_observed) {
  runtime_message <- conditionMessage(runtime_failure)
  cat("Recorded R error: ", runtime_message, "\n", sep = "")
} else {
  runtime_message <- "No R error: the teaching line was edited, or milk_data already exists."
  cat(runtime_message, "\nKeep the original error example for comparison; put your repair in the live-work area.\n", sep = "")
}

Recorded R error: object 'milk_data' not found


For a live repair, the message needs the error, actual object name and column names. Read the example now; use it in Section 3.7:

````{admonition} You → AI: evidence for repair
:class: student-turn
```text
The code returned: object 'milk_data' not found.
My actual data frame is practice, with columns sample_id,
count_cfu_ml and storage_hours. I will paste its four rows below.

Revise the mean calculation using practice. Leave its missing value
unchanged. Use base R and suggest a hand check. Do not change files.
```
````

Run `runtime-repair` for the reference correction:

In [3]:
runtime_mean <- mean(practice$count_cfu_ml, na.rm = TRUE)
runtime_mean
stopifnot(runtime_mean == (10 + 20 + 30) / 3)
stopifnot(is.na(practice$count_cfu_ml[4]))

[1] 20

**Check:** the recorded values are 10, 20 and 30, so the mean is 20. The fourth value is still NA. The check verifies the repair, rather than just the disappearance of an error message. In the live activity, compare your actual reply with this reference after running and checking it.

## 3.3 Inspect code that runs but answers a different question

```{admonition} Illustrative AI reply
:class: assistant-turn
“I will replace the missing count with zero and calculate the mean.”
```

Run the labeled `faulty-answer` exercise. It changes a separate `working` copy:

In [4]:
# Deliberate error: zero imputation changes this task.
working <- practice
working$count_cfu_ml[is.na(working$count_cfu_ml)] <- 0
bad_mean <- mean(working$count_cfu_ml)
bad_mean

[1] 15

**Pause: 15 is an actual output, but is it the requested result?** The recorded counts total 60 across three samples, so their mean is 20. The proposed code introduced an extra zero.

In [5]:
sum(is.na(practice$count_cfu_ml))
bad_mean == (10 + 20 + 30) / 3

[1] 1

[1] FALSE

The original has one missing entry. The comparison with your hand calculation is FALSE.

## 3.4 Repair the analysis assumption

````{admonition} You → AI: the feedback
:class: student-turn
```text
I requested the mean of recorded counts with no imputation.
Your code changed NA to zero and returned 15.
The recorded counts are 10, 20 and 30: 60 / 3 = 20.

Revise only the summary step using practice. Preserve its missing value.
Report missing_n, recorded_n and mean_recorded. Do not edit input files.
```
````

```{admonition} Illustrative AI repair
:class: assistant-turn
“Use na.rm = TRUE to summarize the recorded entries, and report missingness separately. This does not estimate the unrecorded count.”
```

Run `repaired-answer`:

In [6]:
missing_n <- sum(is.na(practice$count_cfu_ml))
recorded_n <- sum(!is.na(practice$count_cfu_ml))
mean_recorded <- mean(practice$count_cfu_ml, na.rm = TRUE)
summary_table <- data.frame(missing_n, recorded_n, mean_recorded)
summary_table

missing_n,recorded_n,mean_recorded
<int>,<int>,<dbl>
1,3,20


**Read the result:** 1 missing, 3 recorded, mean 20. It describes the three recorded samples.

## 3.5 Decide whether to accept it

**Supplied block — `verify-repair`.** Run it to compare with the hand calculation and check that the source NA is preserved:

In [7]:
stopifnot(missing_n == 1, recorded_n == 3, mean_recorded == (10 + 20 + 30) / 3)
stopifnot(is.na(practice$count_cfu_ml[4]))
cat("The requested summary passes; the source observation is still missing.\n")

The requested summary passes; the source observation is still missing.


These checks support accepting the summary **for the stated task**. They do not tell us why the fourth count is missing.

| What happened? | Evidence needed for the repair |
|---|---|
| R stopped: `milk_data` was not found | Actual error text, object name and columns; then a successful calculation and a hand check |
| R returned 15 after replacing NA with zero | The original request, unchanged input and hand-calculated mean 20; “it ran” is insufficient |

| If your live reply… | Your next step |
|---|---|
| Already gives a suitable summary | Verify it; use the faulty reply above as a separate exercise |
| Uses another object or column | Send the actual names and request a localized repair |
| Makes a new data assumption | Ask it to justify the assumption against your task |
| Uses different code but gives 1, 3 and 20 | Check that it also preserves the source and describes the recorded samples |

## 3.6 Keep the result and the evidence

**Supplied block — `save-evidence`.** Run it, then open **Files → outputs → ch03**:

In [8]:
dir.create("outputs/ch03", recursive = TRUE, showWarnings = FALSE)
write.csv(summary_table, "outputs/ch03/summary.csv", row.names = FALSE)
writeLines(c(paste("R error observed:", runtime_error_observed),
             paste("Recorded condition:", runtime_message),
             "Correct object: practice",
             paste("Repaired recorded-count mean:", runtime_mean)),
           "outputs/ch03/runtime-error-and-repair.txt")
if (!file.exists("outputs/ch03/ai-use-log.txt")) {
  file.copy("templates/ai-use-log.txt", "outputs/ch03/ai-use-log.txt")
}
file.exists("outputs/ch03/summary.csv")

[1] TRUE

- **`summary.csv`:** find 1, 3 and 20.
- **`runtime-error-and-repair.txt`:** find the captured R error, correct object name and repaired mean 20.
- **`ai-use-log.txt`:** open as text, fill in your actual route, accepted change and checks, then save. Rerunning preserves an existing log.

```{admonition} Example log for the supplied dialogue
:class: tip dropdown
Route: supplied teaching exercises. Task: repair an object-name error and summarize recorded counts without imputation. Accepted changes: use practice; remove the zero substitution. Evidence: captured R error, then mean 20 = 60/3, with the source NA unchanged. Remaining question: why the fourth result is missing. No live chat was used on this route.
```

**Ready to continue?** Explain what stopped the first calculation, why 15 answered a different question, and how you checked both repairs. Show the saved files. In a live-chat log, record the response you actually received, including any clarification questions.

## 3.7 Try a live response in its own area

**Optional live activity:** choose a chat through [AI access](01-first-run.ipynb#id-1-5-ai-access). If chat is unavailable, your completed supplied exercises still form the code-inspection route.

Find `# %% live-workspace` at the end of your **my-** script. The three comments below mark places to add your own R code. Comments alone do not calculate anything:

In [9]:
# My live AI attempt: summary
# Add only your AI summary code here in your my- copy.
# My live AI attempt: object-name repair
# Add only your AI object-name repair here in your my- copy.
# My live AI attempt: missing-value repair
# Add only your AI missing-value repair here in your my- copy.

1. Send the Section 3.1 request plus `names(practice)` and the four-row `practice` output. Paste only the returned R code below **My live AI attempt: summary**. Save, select only the added lines and Run. Check 1 missing, 3 recorded, mean 20 and the unchanged NA.
2. To practise an object-name repair, use the actual error from Section 3.2 and its repair request. Paste the reply below **My live AI attempt: object-name repair**. Run only those added lines; check a mean of 20. Keep the original `runtime-error` block intact.
3. To practise the missing-value repair, use Section 3.4's feedback request. The zero substitution is our deliberately wrong teaching example; only describe it as an AI mistake if your actual reply made that mistake. Put the returned code below **My live AI attempt: missing-value repair**, then check the original NA and mean 20 again.

If AI already supplied suitable code, record that result rather than inventing a failure. Append the actual prompt, response, check and any repair to `outputs/ch03/ai-use-log.txt`. A different object name or summary layout is acceptable if the numbers and data treatment agree with the task. After a restart, run the reference blocks again before your added code, so `practice` exists.

**Next:** [six farm records you can check by hand](04-small-farm-example.ipynb).